# 🌍 City Quality of Life — Dataset Merging

Produces two clean, merged datasets:
- **`data/merged/cities.csv`** — one row per city, city-level attributes only
- **`data/merged/countries.csv`** — one row per country, country-level attributes only

### Merging rules
1. Numbered/coded column names are renamed to meaningful strings before any merge
2. When the same attribute exists in multiple datasets, the **most recent year wins** (city-level preferred over country-level for the city dataset)
3. Similar attribute names across datasets are unified under a single canonical name
4. Missing values are left as `NaN`
5. Every city / country is kept regardless of coverage

## 0. Setup

In [1]:
!pip install pandas pycountry fuzzywuzzy python-Levenshtein --quiet

In [2]:
import re
import warnings
import numpy as np
import pandas as pd
import pycountry
from pathlib import Path
from fuzzywuzzy import process as fuzz_process

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 10)

RAW_DIR    = Path("data/raw")
MERGED_DIR = Path("data/merged")
MERGED_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Setup complete")

✅ Setup complete


## 1. Shared Utilities

In [3]:
# ── Country name normalisation ────────────────────────────────────────────────
# Manual overrides for names that pycountry can't match automatically
COUNTRY_ALIASES = {
    "United States":              "United States of America",
    "USA":                        "United States of America",
    "US":                         "United States of America",
    "UK":                         "United Kingdom",
    "Russia":                     "Russian Federation",
    "South Korea":                "Korea, Republic of",
    "North Korea":                "Korea, Democratic People's Republic of",
    "Iran":                       "Iran, Islamic Republic of",
    "Syria":                      "Syrian Arab Republic",
    "Tanzania":                   "Tanzania, United Republic of",
    "Bolivia":                    "Bolivia, Plurinational State of",
    "Venezuela":                  "Venezuela, Bolivarian Republic of",
    "Vietnam":                    "Viet Nam",
    "Laos":                       "Lao People's Democratic Republic",
    "Moldova":                    "Moldova, Republic of",
    "Macedonia":                  "North Macedonia",
    "Taiwan":                     "Taiwan, Province of China",
    "Hong Kong":                  "Hong Kong",
    "Trinidad And Tobago":        "Trinidad and Tobago",
    "Bosnia And Herzegovina":     "Bosnia and Herzegovina",
    "Czech Republic":             "Czechia",
    "Ivory Coast":                "Côte d'Ivoire",
    "Cape Verde":                 "Cabo Verde",
    "Swaziland":                  "Eswatini",
    "Kosovo (Disputed Territory)": "Kosovo",
}

def normalise_country(name: str) -> str:
    """Resolve a free-text country name to the pycountry official name (or the input if not found)."""
    if not isinstance(name, str):
        return np.nan
    name = name.strip()
    # Apply manual overrides first
    resolved = COUNTRY_ALIASES.get(name, name)
    # Try exact pycountry lookup
    try:
        return pycountry.countries.lookup(resolved).name
    except LookupError:
        # Fuzzy fallback — only accept high-confidence matches
        candidates = [c.name for c in pycountry.countries]
        match, score = fuzz_process.extractOne(resolved, candidates)
        return match if score >= 85 else resolved


# ── City name splitting ───────────────────────────────────────────────────────
def split_city_field(series: pd.Series) -> pd.DataFrame:
    """
    Splits a combined 'City, State, Country' or 'City, Country' string
    into separate columns: city_name, state (if present), country_name.
    Example inputs:
        'New York, NY, United States'  → city='New York', state='NY', country='United States'
        'Paris, France'                → city='Paris',    state=NaN,  country='France'
        'Tokyo'                        → city='Tokyo',    state=NaN,  country=NaN
    """
    parts = series.str.split(",", expand=True).apply(lambda c: c.str.strip())
    n = parts.shape[1]

    result = pd.DataFrame(index=series.index)
    result["city_name"] = parts[0]

    if n == 1:
        result["state"]        = np.nan
        result["country_name"] = np.nan
    elif n == 2:
        # Could be 'City, Country' or 'City, State' — use heuristic:
        # if second part is a 2-letter uppercase code → state, else country
        is_state = parts[1].str.match(r'^[A-Z]{2}$', na=False)
        result["state"]        = parts[1].where(is_state, np.nan)
        result["country_name"] = parts[1].where(~is_state, np.nan)
    else:
        # 3+ parts: 'City, State, Country'
        result["state"]        = parts[1]
        result["country_name"] = parts[2]

    result["country_name"] = result["country_name"].apply(
        lambda x: normalise_country(x) if pd.notna(x) else np.nan
    )
    return result


# ── Merge helper: prefer most-recent non-null values ─────────────────────────
def coalesce_merge(base: pd.DataFrame, incoming: pd.DataFrame,
                   on: list, incoming_year: int) -> pd.DataFrame:
    """
    Left-join `incoming` onto `base` on `on` keys.
    For overlapping columns:
      - If incoming_year is newer than the existing _year suffix → use incoming value
      - Otherwise keep base value
    Tracks data year per column in a parallel `<col>__year` column.
    """
    overlap = [c for c in incoming.columns if c in base.columns and c not in on]
    # Suffix incoming overlap columns to distinguish during merge
    incoming_renamed = incoming.rename(columns={c: f"{c}__new" for c in overlap})
    merged = base.merge(incoming_renamed, on=on, how="left")

    for col in overlap:
        year_col = f"{col}__year"
        existing_year = merged[year_col].max() if year_col in merged.columns else 0
        new_col = f"{col}__new"

        if incoming_year >= existing_year:
            # Prefer incoming (more recent), fall back to base if incoming is NaN
            merged[col] = merged[new_col].combine_first(merged[col])
            merged[year_col] = np.where(merged[new_col].notna(), incoming_year,
                                        merged.get(year_col, 0))
        else:
            # Keep base; fill NaNs from incoming
            merged[col] = merged[col].combine_first(merged[new_col])
        merged.drop(columns=[new_col], inplace=True)

    # New columns in incoming (no overlap) — just add them
    new_cols = [c for c in incoming.columns if c not in base.columns and c not in on]
    if new_cols:
        for col in new_cols:
            year_col = f"{col}__year"
            merged[year_col] = np.where(merged[col].notna(), incoming_year, np.nan)

    return merged


print("✅ Utilities ready")

✅ Utilities ready


---
# PART A — CITY DATASET
---
## A1. Load & Rename: `qol_index_2024` (Numbeo QoL Index 2024)

In [4]:
df_qol = pd.read_csv(next((RAW_DIR / "qol_index_2024").glob("*.csv")))
print("Original columns:", list(df_qol.columns))
df_qol.head(3)

Original columns: ['Rank', 'City', 'Country', 'Quality of Life Index', 'Purchasing Power Index', 'Safety Index', 'Health Care Index', 'Cost of Living Index', 'Property Price to Income Ratio', 'Traffic Commute Time Index', 'Pollution Index', 'Climate Index']


,Rank,City,Country,Quality of Life Index,Purchasing Power Index,Safety Index,Health Care Index,Cost of Living Index,Property Price to Income Ratio,Traffic Commute Time Index,Pollution Index,Climate Index
0,1,The Hague,Netherlands,223.8,138.9,79.8,80.7,60.2,5.9,21.0,17.5,90.6
1,2,Luxembourg,Luxembourg,220.6,175.9,71.4,76.6,62.0,9.2,27.7,21.6,82.6
2,3,Eindhoven,Netherlands,217.5,139.4,78.1,78.9,62.2,6.0,24.5,19.2,85.4


In [5]:
# Split city field, normalise country, rename index columns
city_parts = split_city_field(df_qol["City"])
df_qol = pd.concat([city_parts, df_qol.drop(columns=["City"])], axis=1)

df_qol = df_qol.rename(columns={
    "Quality of Life Index":           "qol_index",
    "Purchasing Power Index":          "purchasing_power_index",
    "Safety Index":                    "safety_index",
    "Health Care Index":               "healthcare_index",
    "Cost of Living Index":            "cost_of_living_index",
    "Property Price to Income Ratio":  "property_price_to_income_ratio",
    "Traffic Commute Time Index":      "traffic_commute_time_index",
    "Pollution Index":                 "pollution_index",
    "Climate Index":                   "climate_index",
    # Some versions use slightly different names — catch both
    "Rank":                            "qol_rank_2024",
})

print(f"Shape: {df_qol.shape}")
df_qol.head(3)

Shape: (150, 14)


,city_name,state,country_name,qol_rank_2024,Country,qol_index,purchasing_power_index,safety_index,healthcare_index,cost_of_living_index,property_price_to_income_ratio,traffic_commute_time_index,pollution_index,climate_index
0,The Hague,NaN,NaN,1,Netherlands,223.8,138.9,79.8,80.7,60.2,5.9,21.0,17.5,90.6
1,Luxembourg,NaN,NaN,2,Luxembourg,220.6,175.9,71.4,76.6,62.0,9.2,27.7,21.6,82.6
2,Eindhoven,NaN,NaN,3,Netherlands,217.5,139.4,78.1,78.9,62.2,6.0,24.5,19.2,85.4


## A2. Load & Rename: `net_salary` (Average Net Salary)

In [6]:
df_salary = pd.read_csv(next((RAW_DIR / "net_salary").glob("*.csv")))
print("Original columns:", list(df_salary.columns))
df_salary.head(3)

Original columns: ['City', ' "Region"', ' "Country"', ' "Average Monthly Net Salary (After Tax)', ' 2010"', ' "Average Monthly Net Salary (After Tax).1', ' 2011"', ' "Average Monthly Net Salary (After Tax).2', ' 2012"', ' "Average Monthly Net Salary (After Tax).3', ' 2013"', ' "Average Monthly Net Salary (After Tax).4', ' 2014"', ' "Average Monthly Net Salary (After Tax).5', ' 2015"', ' "Average Monthly Net Salary (After Tax).6', ' 2016"', ' "Average Monthly Net Salary (After Tax).7', ' 2017"', ' "Average Monthly Net Salary (After Tax).8', ' 2018"', ' "Average Monthly Net Salary (After Tax).9', ' 2019"', ' "Average Monthly Net Salary (After Tax).10', ' 2020"']


,City,"""Region""","""Country""","""Average Monthly Net Salary (After Tax)","2010""","""Average Monthly Net Salary (After Tax).1","2011""","""Average Monthly Net Salary (After Tax).2","2012""","""Average Monthly Net Salary (After Tax).3","2013""","""Average Monthly Net Salary (After Tax).4","2014""","""Average Monthly Net Salary (After Tax).5","2015""","""Average Monthly Net Salary (After Tax).6","2016""","""Average Monthly Net Salary (After Tax).7","2017""","""Average Monthly Net Salary (After Tax).8","2018""","""Average Monthly Net Salary (After Tax).9","2019""","""Average Monthly Net Salary (After Tax).10","2020"""
0,New York City,"""New York""","""United States of America""",3300.0,3568.530000,4017.991667,4339.513069,3476.340323,3376.010678,4026.656605,4121.088445,4658.585654,5122.297026,6023.253437,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Washington, D.C.","""District of Columbia""","""United States of America""",2200.0,3365.200744,3700.000000,3981.700000,4573.356667,3479.050000,3900.222222,4095.370370,4762.834321,5596.545455,5601.921903,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,San Francisco,"""California""","""United States of America""",4700.0,2633.333333,4195.428571,4167.538462,3470.000000,4058.783333,4281.510400,3975.943333,6111.775926,7289.561905,7930.791453,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# Detect the city and salary columns dynamically
city_col   = [c for c in df_salary.columns if "city" in c.lower()][0]
salary_col = [c for c in df_salary.columns if "salary" in c.lower() or "net" in c.lower() or "wage" in c.lower()][0]

city_parts = split_city_field(df_salary[city_col])
df_salary  = pd.concat([city_parts, df_salary.drop(columns=[city_col])], axis=1)
df_salary  = df_salary.rename(columns={salary_col: "avg_monthly_net_salary_usd"})

# Keep only city keys + salary
df_salary = df_salary[["city_name", "state", "country_name", "avg_monthly_net_salary_usd"]]
print(f"Shape: {df_salary.shape}")
df_salary.head(3)

Shape: (685, 4)


,city_name,state,country_name,avg_monthly_net_salary_usd
0,New York City,NaN,NaN,3300.0
1,Washington,NaN,Turks and Caicos Islands,2200.0
2,San Francisco,NaN,NaN,4700.0


## A3. Load & Rename: `qol_dataset` (Teleport QoL — 17 categories)

In [8]:
df_teleport = pd.read_csv(next((RAW_DIR / "qol_dataset").glob("*.csv")))
print("Original columns:", list(df_teleport.columns))
df_teleport.head(3)

Original columns: ['Unnamed: 0', 'UA_Name', 'UA_Country', 'UA_Continent', 'Housing', 'Cost of Living', 'Startups', 'Venture Capital', 'Travel Connectivity', 'Commute', 'Business Freedom', 'Safety', 'Healthcare', 'Education', 'Environmental Quality', 'Economy', 'Taxation', 'Internet Access', 'Leisure & Culture', 'Tolerance', 'Outdoors']


,Unnamed: 0,UA_Name,UA_Country,UA_Continent,Housing,Cost of Living,Startups,Venture Capital,Travel Connectivity,Commute,Business Freedom,Safety,Healthcare,Education,Environmental Quality,Economy,Taxation,Internet Access,Leisure & Culture,Tolerance,Outdoors
0,0,Aarhus,Denmark,Europe,6.1315,4.015,2.8270,2.512,3.5360,6.31175,9.940000,9.6165,8.704333,5.3665,7.63300,4.8865,5.0680,8.373,3.1870,9.7385,4.1300
1,1,Adelaide,Australia,Oceania,6.3095,4.692,3.1365,2.640,1.7765,5.33625,9.399667,7.9260,7.936667,5.1420,8.33075,6.0695,4.5885,4.341,4.3285,7.8220,5.5310
2,2,Albuquerque,New Mexico,North America,7.2620,6.059,3.7720,1.493,1.4555,5.05575,8.671000,1.3435,6.430000,4.1520,7.31950,6.5145,4.3460,5.396,4.8900,7.0285,3.5155


In [9]:
# Teleport dataset has 17 category scores + city name
# Canonical renaming (snake_case, descriptive)
TELEPORT_RENAME = {
    "Housing":             "teleport_housing_score",
    "Cost of Living":      "teleport_cost_of_living_score",
    "Startups":            "teleport_startups_score",
    "Venture Capital":     "teleport_venture_capital_score",
    "Travel Connectivity": "teleport_travel_connectivity_score",
    "Commute":             "teleport_commute_score",
    "Business Freedom":    "teleport_business_freedom_score",
    "Safety":              "teleport_safety_score",
    "Healthcare":          "teleport_healthcare_score",
    "Education":           "teleport_education_score",
    "Environmental Quality": "teleport_environmental_quality_score",
    "Economy":             "teleport_economy_score",
    "Taxation":            "teleport_taxation_score",
    "Internet Access":     "teleport_internet_access_score",
    "Leisure & Culture":   "teleport_leisure_culture_score",
    "Tolerance":           "teleport_tolerance_score",
    "Outdoors":            "teleport_outdoors_score",
    # City column names vary — catch common variants
    "city_name": "city_name",
    "City":      "city_name",
    "ua_name":   "city_name",
}

df_teleport = df_teleport.rename(columns=TELEPORT_RENAME)

# If city is not yet split, split it
if "city_name" in df_teleport.columns:
    city_parts  = split_city_field(df_teleport["city_name"])
    df_teleport = pd.concat([city_parts, df_teleport.drop(columns=["city_name"])], axis=1)

print(f"Shape: {df_teleport.shape}")
df_teleport.head(3)

Shape: (266, 21)


,Unnamed: 0,UA_Name,UA_Country,UA_Continent,teleport_housing_score,teleport_cost_of_living_score,teleport_startups_score,teleport_venture_capital_score,teleport_travel_connectivity_score,teleport_commute_score,teleport_business_freedom_score,teleport_safety_score,teleport_healthcare_score,teleport_education_score,teleport_environmental_quality_score,teleport_economy_score,teleport_taxation_score,teleport_internet_access_score,teleport_leisure_culture_score,teleport_tolerance_score,teleport_outdoors_score
0,0,Aarhus,Denmark,Europe,6.1315,4.015,2.8270,2.512,3.5360,6.31175,9.940000,9.6165,8.704333,5.3665,7.63300,4.8865,5.0680,8.373,3.1870,9.7385,4.1300
1,1,Adelaide,Australia,Oceania,6.3095,4.692,3.1365,2.640,1.7765,5.33625,9.399667,7.9260,7.936667,5.1420,8.33075,6.0695,4.5885,4.341,4.3285,7.8220,5.5310
2,2,Albuquerque,New Mexico,North America,7.2620,6.059,3.7720,1.493,1.4555,5.05575,8.671000,1.3435,6.430000,4.1520,7.31950,6.5145,4.3460,5.396,4.8900,7.0285,3.5155


## A4. Load & Rename: `cost_of_living` (Numbeo 55-item prices)

In [10]:
df_col = pd.read_csv(next((RAW_DIR / "cost_of_living").glob("*.csv")))
print("Original columns:", list(df_col.columns))
df_col.head(3)

Original columns: ['city', 'country', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9', 'x10', 'x11', 'x12', 'x13', 'x14', 'x15', 'x16', 'x17', 'x18', 'x19', 'x20', 'x21', 'x22', 'x23', 'x24', 'x25', 'x26', 'x27', 'x28', 'x29', 'x30', 'x31', 'x32', 'x33', 'x34', 'x35', 'x36', 'x37', 'x38', 'x39', 'x40', 'x41', 'x42', 'x43', 'x44', 'x45', 'x46', 'x47', 'x48', 'x49', 'x50', 'x51', 'x52', 'x53', 'x54', 'x55', 'data_quality']


,city,country,x1,x2,x3,x4,x5,x6,x7,x8,x9,x10,x11,x12,x13,x14,x15,x16,x17,x18,x19,x20,x21,x22,x23,x24,x25,x26,x27,x28,x29,x30,x31,x32,x33,x34,x35,x36,x37,x38,x39,x40,x41,x42,x43,x44,x45,x46,x47,x48,x49,x50,x51,x52,x53,x54,x55,data_quality
0,Seoul,South Korea,7.68,53.78,6.15,3.07,4.99,3.93,1.48,0.79,2.20,2.85,3.53,4.04,11.54,10.58,41.61,6.77,3.71,6.50,6.19,3.84,2.92,2.45,1.05,15.36,2.12,2.36,3.46,1.00,42.25,2.92,0.92,9.22,1.43,29192.75,27840.35,182.13,0.16,22.48,55.88,18.33,9.60,404.15,17902.55,58.26,46.36,70.81,110.36,742.54,557.52,2669.12,1731.08,22067.70,10971.90,2689.62,3.47,1
1,Shanghai,China,5.69,39.86,5.69,1.14,4.27,3.98,0.53,0.33,2.74,2.61,1.22,2.22,18.35,4.86,13.12,2.26,1.60,2.19,1.53,0.84,1.04,0.83,0.64,14.24,0.94,1.97,3.56,0.57,28.47,2.14,0.43,8.54,1.20,19929.68,20750.95,66.00,0.03,17.07,63.49,14.95,8.54,1382.62,26379.45,70.49,34.92,88.21,123.51,1091.93,569.88,2952.70,1561.59,17746.11,9416.35,1419.87,5.03,1
2,Guangzhou,China,4.13,28.47,4.98,0.85,1.71,3.54,0.44,0.33,1.91,1.63,1.03,1.71,9.00,3.77,11.75,2.02,1.44,1.82,1.31,0.74,1.00,0.49,0.51,11.39,0.95,2.26,3.70,0.36,28.47,1.71,0.37,6.26,1.19,21353.23,21381.70,59.65,0.02,16.66,34.17,18.98,8.54,555.18,24556.21,63.43,33.83,66.73,43.89,533.28,317.45,1242.24,688.05,12892.82,5427.45,1211.68,5.19,1


In [11]:
# Numbeo item_id → descriptive column name mapping
# Source: Numbeo API documentation (https://www.numbeo.com/common/api.jsp)
NUMBEO_ITEMS = {
    "x1":  "price_meal_inexpensive_restaurant_usd",
    "x2":  "price_meal_midrange_2people_usd",
    "x3":  "price_mcmeal_fastfood_usd",
    "x4":  "price_domestic_beer_draught_usd",
    "x5":  "price_imported_beer_bottle_usd",
    "x6":  "price_cappuccino_usd",
    "x7":  "price_coke_pepsi_usd",
    "x8":  "price_water_bottle_restaurant_usd",
    "x9":  "price_milk_1liter_usd",
    "x10": "price_bread_500g_usd",
    "x11": "price_rice_1kg_usd",
    "x12": "price_eggs_12_usd",
    "x13": "price_local_cheese_1kg_usd",
    "x14": "price_chicken_fillets_1kg_usd",
    "x15": "price_beef_1kg_usd",
    "x16": "price_apples_1kg_usd",
    "x17": "price_banana_1kg_usd",
    "x18": "price_oranges_1kg_usd",
    "x19": "price_tomato_1kg_usd",
    "x20": "price_potato_1kg_usd",
    "x21": "price_onion_1kg_usd",
    "x22": "price_lettuce_1head_usd",
    "x23": "price_water_1_5liter_market_usd",
    "x24": "price_wine_bottle_midrange_usd",
    "x25": "price_domestic_beer_bottle_market_usd",
    "x26": "price_imported_beer_bottle_market_usd",
    "x27": "price_cigarettes_20pack_usd",
    "x28": "price_one_way_transport_ticket_usd",
    "x29": "price_monthly_transport_pass_usd",
    "x30": "price_taxi_start_usd",
    "x31": "price_taxi_1km_usd",
    "x32": "price_taxi_1hr_waiting_usd",
    "x33": "price_gasoline_1liter_usd",
    "x34": "price_volkswagen_golf_new_usd",
    "x35": "price_toyota_corolla_new_usd",
    "x36": "price_basic_utilities_85sqm_usd",
    "x37": "price_mobile_plan_monthly_usd",
    "x38": "price_internet_60mbps_usd",
    "x39": "price_fitness_club_monthly_usd",
    "x40": "price_tennis_court_1hr_usd",
    "x41": "price_cinema_ticket_usd",
    "x42": "price_preschool_monthly_usd",
    "x43": "price_intl_primary_school_yearly_usd",
    "x44": "price_jeans_levis_usd",
    "x45": "price_summer_dress_usd",
    "x46": "price_nike_running_shoes_usd",
    "x47": "price_mens_leather_shoes_usd",
    "x48": "price_rent_1br_city_center_usd",
    "x49": "price_rent_1br_outside_center_usd",
    "x50": "price_rent_3br_city_center_usd",
    "x51": "price_rent_3br_outside_center_usd",
    "x52": "price_sqm_apartment_city_center_usd",
    "x53": "price_sqm_apartment_outside_center_usd",
    "x54": "avg_monthly_net_salary_usd",  # canonical name — will be resolved by coalesce
    "x55": "mortgage_interest_rate_pct",
}

df_col = df_col.rename(columns=NUMBEO_ITEMS)

# Split city field
city_col = [c for c in df_col.columns if c.lower() in ("city", "city_name")][0]
city_parts = split_city_field(df_col[city_col])
df_col = pd.concat([city_parts, df_col.drop(columns=[city_col])], axis=1)

print(f"Shape: {df_col.shape}")
df_col.head(3)

Shape: (4956, 60)


,city_name,state,country_name,country,price_meal_inexpensive_restaurant_usd,price_meal_midrange_2people_usd,price_mcmeal_fastfood_usd,price_domestic_beer_draught_usd,price_imported_beer_bottle_usd,price_cappuccino_usd,price_coke_pepsi_usd,price_water_bottle_restaurant_usd,price_milk_1liter_usd,price_bread_500g_usd,price_rice_1kg_usd,price_eggs_12_usd,price_local_cheese_1kg_usd,price_chicken_fillets_1kg_usd,price_beef_1kg_usd,price_apples_1kg_usd,price_banana_1kg_usd,price_oranges_1kg_usd,price_tomato_1kg_usd,price_potato_1kg_usd,price_onion_1kg_usd,price_lettuce_1head_usd,price_water_1_5liter_market_usd,price_wine_bottle_midrange_usd,price_domestic_beer_bottle_market_usd,price_imported_beer_bottle_market_usd,price_cigarettes_20pack_usd,price_one_way_transport_ticket_usd,price_monthly_transport_pass_usd,price_taxi_start_usd,price_taxi_1km_usd,price_taxi_1hr_waiting_usd,price_gasoline_1liter_usd,price_volkswagen_golf_new_usd,price_toyota_corolla_new_usd,price_basic_utilities_85sqm_usd,price_mobile_plan_monthly_usd,price_internet_60mbps_usd,price_fitness_club_monthly_usd,price_tennis_court_1hr_usd,price_cinema_ticket_usd,price_preschool_monthly_usd,price_intl_primary_school_yearly_usd,price_jeans_levis_usd,price_summer_dress_usd,price_nike_running_shoes_usd,price_mens_leather_shoes_usd,price_rent_1br_city_center_usd,price_rent_1br_outside_center_usd,price_rent_3br_city_center_usd,price_rent_3br_outside_center_usd,price_sqm_apartment_city_center_usd,price_sqm_apartment_outside_center_usd,avg_monthly_net_salary_usd,mortgage_interest_rate_pct,data_quality
0,Seoul,NaN,NaN,South Korea,7.68,53.78,6.15,3.07,4.99,3.93,1.48,0.79,2.20,2.85,3.53,4.04,11.54,10.58,41.61,6.77,3.71,6.50,6.19,3.84,2.92,2.45,1.05,15.36,2.12,2.36,3.46,1.00,42.25,2.92,0.92,9.22,1.43,29192.75,27840.35,182.13,0.16,22.48,55.88,18.33,9.60,404.15,17902.55,58.26,46.36,70.81,110.36,742.54,557.52,2669.12,1731.08,22067.70,10971.90,2689.62,3.47,1
1,Shanghai,NaN,NaN,China,5.69,39.86,5.69,1.14,4.27,3.98,0.53,0.33,2.74,2.61,1.22,2.22,18.35,4.86,13.12,2.26,1.60,2.19,1.53,0.84,1.04,0.83,0.64,14.24,0.94,1.97,3.56,0.57,28.47,2.14,0.43,8.54,1.20,19929.68,20750.95,66.00,0.03,17.07,63.49,14.95,8.54,1382.62,26379.45,70.49,34.92,88.21,123.51,1091.93,569.88,2952.70,1561.59,17746.11,9416.35,1419.87,5.03,1
2,Guangzhou,NaN,NaN,China,4.13,28.47,4.98,0.85,1.71,3.54,0.44,0.33,1.91,1.63,1.03,1.71,9.00,3.77,11.75,2.02,1.44,1.82,1.31,0.74,1.00,0.49,0.51,11.39,0.95,2.26,3.70,0.36,28.47,1.71,0.37,6.26,1.19,21353.23,21381.70,59.65,0.02,16.66,34.17,18.98,8.54,555.18,24556.21,63.43,33.83,66.73,43.89,533.28,317.45,1242.24,688.05,12892.82,5427.45,1211.68,5.19,1


## A5. Load & Rename: `city_happiness_2024`

In [12]:
df_chappy = pd.read_csv(next((RAW_DIR / "city_happiness_2024").glob("*.csv")))
print("Original columns:", list(df_chappy.columns))
df_chappy.head(3)

Original columns: ['City', 'Month', 'Year', 'Decibel_Level', 'Traffic_Density', 'Green_Space_Area', 'Air_Quality_Index', 'Happiness_Score', 'Cost_of_Living_Index', 'Healthcare_Index']


,City,Month,Year,Decibel_Level,Traffic_Density,Green_Space_Area,Air_Quality_Index,Happiness_Score,Cost_of_Living_Index,Healthcare_Index
0,Auckland,January,2030,55,Low,80,40,8.4,110,97
1,Berlin,January,2030,50,Low,60,45,7.9,80,93
2,Cairo,January,2030,75,Very High,15,110,4.1,55,69


In [13]:
city_col = [c for c in df_chappy.columns if "city" in c.lower()][0]
city_parts = split_city_field(df_chappy[city_col])
df_chappy  = pd.concat([city_parts, df_chappy.drop(columns=[city_col])], axis=1)

# Rename happiness score to canonical name
for col in df_chappy.columns:
    if "happiness" in col.lower() and "score" in col.lower():
        df_chappy = df_chappy.rename(columns={col: "city_happiness_score"})
    elif "rank" in col.lower():
        df_chappy = df_chappy.rename(columns={col: "city_happiness_rank"})

print(f"Shape: {df_chappy.shape}")
df_chappy.head(3)

Shape: (51, 12)


,city_name,state,country_name,Month,Year,Decibel_Level,Traffic_Density,Green_Space_Area,Air_Quality_Index,city_happiness_score,Cost_of_Living_Index,Healthcare_Index
0,Auckland,NaN,NaN,January,2030,55,Low,80,40,8.4,110,97
1,Berlin,NaN,NaN,January,2030,50,Low,60,45,7.9,80,93
2,Cairo,NaN,NaN,January,2030,75,Very High,15,110,4.1,55,69


## A6. Load & Rename: `sunshine_hours` and `avg_temperature`

In [14]:
df_sun  = pd.read_csv(next((RAW_DIR / "sunshine_hours").glob("*.csv")))
df_temp = pd.read_csv(next((RAW_DIR / "avg_temperature").glob("*.csv")))
print("Sunshine columns :", list(df_sun.columns))
print("Temperature columns:", list(df_temp.columns))

Sunshine columns : ['Country', 'City', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'Year']
Temperature columns: ['Country', 'City', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'Year', 'Ref.']


In [17]:
def prep_climate_df(df, annual_col_hint, annual_name):
    """Generic prep for sunshine / temperature datasets."""
    city_col = [c for c in df.columns if "city" in c.lower()][0]
    city_parts = split_city_field(df[city_col])
    df = pd.concat([city_parts, df.drop(columns=[city_col])], axis=1)

    # Rename month columns to consistent names
    month_map = {
        "jan": "jan", "feb": "feb", "mar": "mar", "apr": "apr",
        "may": "may", "jun": "jun", "jul": "jul", "aug": "aug",
        "sep": "sep", "oct": "oct", "nov": "nov", "dec": "dec"
    }
    rename = {}
    for col in df.columns:
        lower = col.lower()
        for abbr, std in month_map.items():
            if lower.startswith(abbr):
                rename[col] = f"{annual_col_hint}_{std}"
                break
        if annual_col_hint in lower or "annual" in lower or "yearly" in lower or "avg" in lower:
            rename[col] = annual_name
    df = df.rename(columns=rename)
    return df


df_sun  = prep_climate_df(df_sun,  "sunshine_hours", "annual_sunshine_hours")
df_temp = prep_climate_df(df_temp, "avg_temp_c",     "annual_avg_temp_c")

print(f"Sunshine shape:     {df_sun.shape}")
print(f"Temperature shape:  {df_temp.shape}")
df_sun.head(2)

Sunshine shape:     (392, 19)
Temperature shape:  (465, 20)


,city_name,state,country_name,state,country_name,Country,annual_sunshine_hours,annual_sunshine_hours,annual_sunshine_hours,annual_sunshine_hours,annual_sunshine_hours,annual_sunshine_hours,annual_sunshine_hours,annual_sunshine_hours,annual_sunshine_hours,annual_sunshine_hours,annual_sunshine_hours,annual_sunshine_hours,Year
0,Gagnoa,NaN,NaN,NaN,NaN,Ivory Coast,183.0,180.0,196.0,188.0,181.0,118.0,97.0,80.0,110.0,155.0,171.0,164.0,1823.0
1,Bouaké,NaN,NaN,NaN,NaN,Ivory Coast,242.0,224.0,219.0,194.0,208.0,145.0,104.0,82.0,115.0,170.0,191.0,198.0,2092.0


## A7. Load & Rename: `air_quality_2024`

In [18]:
df_aqi = pd.read_csv(next((RAW_DIR / "air_quality_2024").glob("*.csv")), sep=";")
print("Original columns:", list(df_aqi.columns))
df_aqi.head(3)

Original columns: ['Country Code', 'City', 'Location', 'Coordinates', 'Pollutant', 'Source Name', 'Unit', 'Value', 'Last Updated', 'Country Label']


,Country Code,City,Location,Coordinates,Pollutant,Source Name,Unit,Value,Last Updated,Country Label
0,JP,NaN,北九州市小倉北区大門一丁目６－４８,"33.880833, 130.873056",NO,japan-soramame,ppm,0.002,2024-03-10T13:30:00+05:30,Japan
1,JP,NaN,北九州市若松区本町三丁目１３－１,"33.898056, 130.81",NO2,japan-soramame,ppm,0.005,2024-03-10T13:30:00+05:30,Japan
2,JP,NaN,北九州市門司区大里原町１２－１２,"33.895833, 130.935833",NOX,japan-soramame,ppm,0.013,2024-03-10T13:30:00+05:30,Japan


In [21]:
# The AQI CSV is semicolon-separated, long-format (one row per pollutant reading).
# Columns: Country Code, City, Location, Coordinates, Pollutant, Source Name, Unit, Value, Last Updated, Country Label

# Rename to canonical names
df_aqi = df_aqi.rename(columns={
    "City":          "city_name",
    "Country Label": "country_name",
})

# Drop rows without a city name
df_aqi = df_aqi.dropna(subset=["city_name"])
df_aqi = df_aqi[df_aqi["city_name"].str.strip() != ""]

# Normalise country
df_aqi["country_name"] = df_aqi["country_name"].apply(normalise_country)

# Pivot: one row per (city, country) with average Value per pollutant
# Map pollutant names → canonical column names
POLLUTANT_MAP = {
    "PM2.5": "pm25_aqi_value",
    "PM10":  "pm10_aqi_value",
    "NO2":   "no2_aqi_value",
    "O3":    "ozone_aqi_value",
    "CO":    "co_aqi_value",
    "SO2":   "so2_aqi_value",
    "NO":    "no_aqi_value",
    "NOX":   "nox_aqi_value",
}
df_aqi["Pollutant"] = df_aqi["Pollutant"].map(POLLUTANT_MAP)
df_aqi = df_aqi.dropna(subset=["Pollutant"])

# Average each pollutant per city-country pair
df_aqi = (df_aqi
          .groupby(["city_name", "country_name", "Pollutant"])["Value"]
          .mean()
          .reset_index()
          .pivot_table(index=["city_name", "country_name"],
                       columns="Pollutant", values="Value")
          .reset_index())
df_aqi.columns.name = None   # remove the pivot MultiIndex name

# Add state column (not available in this dataset)
df_aqi["state"] = np.nan

print(f"Shape: {df_aqi.shape}")
df_aqi.head(3)


Shape: (2457, 11)


,city_name,country_name,co_aqi_value,no2_aqi_value,no_aqi_value,nox_aqi_value,ozone_aqi_value,pm10_aqi_value,pm25_aqi_value,so2_aqi_value,state
0,007,United States,NaN,NaN,NaN,NaN,NaN,NaN,5.528571,NaN,NaN
1,019,United States,NaN,NaN,NaN,NaN,NaN,NaN,3.400000,NaN,NaN
2,037,United States,NaN,NaN,NaN,NaN,NaN,NaN,18.356522,NaN,NaN


In [19]:
AQI_RENAME = {
    # Common Kaggle AQI column naming variants → canonical
    "City":           "city_name",
    "city":           "city_name",
    "Country":        "country_name",
    "country":        "country_name",
    "AQI Value":      "aqi_value",
    "AQI Category":   "aqi_category",
    "CO AQI Value":   "co_aqi_value",
    "Ozone AQI Value":"ozone_aqi_value",
    "NO2 AQI Value":  "no2_aqi_value",
    "PM2.5 AQI Value":"pm25_aqi_value",
    "Country Label":  "country_name"
}
df_aqi = df_aqi.rename(columns=AQI_RENAME)

if "city_name" in df_aqi.columns:
    city_parts = split_city_field(df_aqi["city_name"])
    # Only use the split result for city_name; keep existing country_name if present
    df_aqi["city_name"] = city_parts["city_name"]
    if "country_name" not in df_aqi.columns:
        df_aqi["country_name"] = city_parts["country_name"]
    df_aqi["state"] = city_parts["state"]

if "country_name" in df_aqi.columns:
    df_aqi["country_name"] = df_aqi["country_name"].apply(normalise_country)

print(f"Shape: {df_aqi.shape}")
df_aqi.head(3)

Shape: (54255, 11)


,Country Code,city_name,Location,Coordinates,Pollutant,Source Name,Unit,Value,Last Updated,country_name,state
0,JP,NaN,北九州市小倉北区大門一丁目６－４８,"33.880833, 130.873056",NO,japan-soramame,ppm,0.002,2024-03-10T13:30:00+05:30,Japan,NaN
1,JP,NaN,北九州市若松区本町三丁目１３－１,"33.898056, 130.81",NO2,japan-soramame,ppm,0.005,2024-03-10T13:30:00+05:30,Japan,NaN
2,JP,NaN,北九州市門司区大里原町１２－１２,"33.895833, 130.935833",NOX,japan-soramame,ppm,0.013,2024-03-10T13:30:00+05:30,Japan,NaN


## A8. Merge All City-Level Datasets

In [22]:
CITY_KEYS = ["city_name", "country_name"]

# Dataset list ordered by year (oldest first so newer data overwrites)
# Format: (dataframe, year, label)
CITY_SOURCES = [
    (df_teleport, 2021, "Teleport QoL"),
    (df_col,      2022, "Cost of Living (Numbeo)"),
    (df_salary,   2022, "Net Salary"),
    (df_sun,      2023, "Sunshine Hours"),
    (df_temp,     2023, "Average Temperature"),
    (df_chappy,   2024, "City Happiness"),
    (df_aqi,      2024, "Air Quality"),
    (df_qol,      2024, "Numbeo QoL Index"),
]

# Start with a union of all city keys
all_city_keys = pd.concat(
    [df[CITY_KEYS].dropna(subset=["city_name"]) for df, _, _ in CITY_SOURCES],
    ignore_index=True
).drop_duplicates()

cities = all_city_keys.copy()
print(f"Unique city-country pairs: {len(cities)}")

KeyError: "None of [Index(['city_name', 'country_name'], dtype='object')] are in the [columns]"

In [ ]:
for df, year, label in CITY_SOURCES:
    # Ensure keys are present
    if not all(k in df.columns for k in CITY_KEYS):
        print(f"  ⚠️  Skipping {label} — missing key columns")
        continue

    # Drop rows without city_name
    df_clean = df.dropna(subset=["city_name"]).copy()

    cities = coalesce_merge(cities, df_clean, on=CITY_KEYS, incoming_year=year)
    print(f"  ✅  {label} ({year})  → {cities.shape}")

print(f"\nFinal city dataset: {cities.shape[0]:,} rows × {cities.shape[1]} cols")

In [ ]:
# Drop internal __year tracking columns from final output
year_cols = [c for c in cities.columns if c.endswith("__year")]
cities_clean = cities.drop(columns=year_cols)

# Reorder: keys first
key_cols   = ["city_name", "state", "country_name"]
other_cols = [c for c in cities_clean.columns if c not in key_cols]
cities_clean = cities_clean[key_cols + other_cols]

# Save
out_path = MERGED_DIR / "cities.csv"
cities_clean.to_csv(out_path, index=False)
print(f"✅ Saved: {out_path}")
print(f"   Shape : {cities_clean.shape[0]:,} rows × {cities_clean.shape[1]} cols")
print(f"   Columns: {list(cities_clean.columns)}")
cities_clean.head(5)

---
# PART B — COUNTRY DATASET
---
## B1. Load & Rename: `happiness_2024` (World Happiness Report)

In [ ]:
df_happy = pd.read_csv(next((RAW_DIR / "happiness_2024").glob("*.csv")))
print("Original columns:", list(df_happy.columns))
df_happy.head(3)

In [ ]:
HAPPY_RENAME = {
    "Country name":                "country_name",
    "Country":                     "country_name",
    "Regional indicator":          "world_region",
    "Ladder score":                "happiness_ladder_score",
    "Logged GDP per capita":       "log_gdp_per_capita",
    "Log GDP per capita":          "log_gdp_per_capita",
    "Social support":              "social_support_score",
    "Healthy life expectancy":     "healthy_life_expectancy_years",
    "Freedom to make life choices":"freedom_score",
    "Generosity":                  "generosity_score",
    "Perceptions of corruption":   "corruption_perception_score",
    "Happiness Rank":              "happiness_rank",
    "Happiness Score":             "happiness_ladder_score",
    "Score":                       "happiness_ladder_score",
}
df_happy = df_happy.rename(columns=HAPPY_RENAME)
df_happy["country_name"] = df_happy["country_name"].apply(normalise_country)

# Keep only country-level attributes
keep = ["country_name", "world_region", "happiness_ladder_score",
        "log_gdp_per_capita", "social_support_score",
        "healthy_life_expectancy_years", "freedom_score",
        "generosity_score", "corruption_perception_score", "happiness_rank"]
df_happy = df_happy[[c for c in keep if c in df_happy.columns]]
print(f"Shape: {df_happy.shape}")
df_happy.head(3)

## B2. Load & Rename: `internet_speed_2024`

In [ ]:
df_inet = pd.read_csv(next((RAW_DIR / "internet_speed_2024").glob("*.csv")))
print("Original columns:", list(df_inet.columns))
df_inet.head(3)

In [ ]:
INET_RENAME = {
    "Country":                     "country_name",
    "country":                     "country_name",
    "Avg. (Mb/s)": "avg_broadband_speed_mbps",
    "Average (Mb/s)": "avg_broadband_speed_mbps",
    "Broadband Download Speed (Mbps)": "avg_broadband_speed_mbps",
    "Download Speed (Mbps)":       "avg_broadband_speed_mbps",
    "avg_download_mbps":           "avg_broadband_speed_mbps",
}
df_inet = df_inet.rename(columns=INET_RENAME)
df_inet["country_name"] = df_inet["country_name"].apply(normalise_country)

# Keep relevant columns
keep = ["country_name"] + [c for c in df_inet.columns
        if c != "country_name" and any(x in c.lower()
        for x in ["speed", "mbps", "broadband", "rank"])]
df_inet = df_inet[keep]
print(f"Shape: {df_inet.shape}")
df_inet.head(3)

## B3. Load & Rename: `lgbtq_index`

In [ ]:
df_lgbtq = pd.read_csv(RAW_DIR / "lgbtq_index" / "lgbtq_legal_equality_index.csv")
print("Original columns:", list(df_lgbtq.columns))
df_lgbtq.head(3)

In [ ]:
LGBTQ_RENAME = {
    "Entity":  "country_name",
    "Country": "country_name",
    "Year":    "lgbtq_data_year",
}
df_lgbtq = df_lgbtq.rename(columns=LGBTQ_RENAME)

# Rename the score column (variable name in OWID datasets)
score_col = [c for c in df_lgbtq.columns if c not in ("country_name", "lgbtq_data_year", "Code")][0]
df_lgbtq  = df_lgbtq.rename(columns={score_col: "lgbtq_legal_equality_index"})

# Keep most recent year per country
if "lgbtq_data_year" in df_lgbtq.columns:
    df_lgbtq = (df_lgbtq.sort_values("lgbtq_data_year", ascending=False)
                         .drop_duplicates(subset=["country_name"])
                         .drop(columns=["lgbtq_data_year", "Code"], errors="ignore"))

df_lgbtq["country_name"] = df_lgbtq["country_name"].apply(normalise_country)
print(f"Shape: {df_lgbtq.shape}")
df_lgbtq.head(3)

## B4. Load & Rename: `numbeo_country_2024` (Healthcare, Crime, Pollution)

In [ ]:
df_numbeo_c = pd.read_csv(RAW_DIR / "numbeo_country_2024" / "numbeo_country_indices_2024.csv")
print("Original columns:", list(df_numbeo_c.columns))
df_numbeo_c.head(3)

In [ ]:
df_numbeo_c["country_name"] = df_numbeo_c["Country"].apply(normalise_country)
df_numbeo_c = df_numbeo_c.drop(columns=["Country"])

# Columns already have index prefix from the scraper (e.g. healthcare_health_care_index)
# Clean them up further
df_numbeo_c = df_numbeo_c.rename(columns={
    "healthcare_health_care_index":         "country_healthcare_index",
    "healthcare_health_care_exp._index":    "country_healthcare_exp_index",
    "crime_crime_index":                    "country_crime_index",
    "crime_safety_index":                   "country_safety_index",
    "pollution_pollution_index":            "country_pollution_index",
    "pollution_exp_pollution_index":        "country_pollution_exp_index",
})
print(f"Shape: {df_numbeo_c.shape}")
df_numbeo_c.head(3)

## B5. Load & Rename: `english_proficiency`

In [ ]:
df_epi = pd.read_csv(RAW_DIR / "english_proficiency" / "ef_english_proficiency_index_2024.csv")
print("Original columns:", list(df_epi.columns))
df_epi.head(3)

In [ ]:
# Detect country and score columns dynamically
country_col = [c for c in df_epi.columns if "country" in c.lower() or "nation" in c.lower()][0]
score_col   = [c for c in df_epi.columns
               if any(x in c.lower() for x in ["score", "epi", "proficiency", "index"])
               and c != country_col][0]

df_epi = df_epi.rename(columns={
    country_col: "country_name",
    score_col:   "english_proficiency_score",
})
df_epi["country_name"] = df_epi["country_name"].apply(normalise_country)
df_epi = df_epi[["country_name", "english_proficiency_score"]].dropna(subset=["country_name"])
print(f"Shape: {df_epi.shape}")
df_epi.head(3)

## B6. Merge All Country-Level Datasets

In [ ]:
COUNTRY_KEY = ["country_name"]

COUNTRY_SOURCES = [
    (df_numbeo_c, 2024, "Numbeo Country Indices"),
    (df_happy,    2024, "World Happiness Report"),
    (df_inet,     2024, "Internet Speed"),
    (df_lgbtq,    2025, "LGBT+ Legal Equality"),
    (df_epi,      2024, "EF English Proficiency"),
]

# Build union of all country keys
all_country_keys = pd.concat(
    [df[COUNTRY_KEY].dropna() for df, _, _ in COUNTRY_SOURCES],
    ignore_index=True
).drop_duplicates()

countries = all_country_keys.copy()
print(f"Unique countries: {len(countries)}")

In [ ]:
for df, year, label in COUNTRY_SOURCES:
    if "country_name" not in df.columns:
        print(f"  ⚠️  Skipping {label} — no country_name column")
        continue
    df_clean = df.dropna(subset=["country_name"]).copy()
    countries = coalesce_merge(countries, df_clean, on=COUNTRY_KEY, incoming_year=year)
    print(f"  ✅  {label} ({year})  → {countries.shape}")

print(f"\nFinal country dataset: {countries.shape[0]:,} rows × {countries.shape[1]} cols")

In [ ]:
# Drop __year tracking columns
year_cols = [c for c in countries.columns if c.endswith("__year")]
countries_clean = countries.drop(columns=year_cols)

# Reorder: key first
other_cols = [c for c in countries_clean.columns if c != "country_name"]
countries_clean = countries_clean[["country_name"] + other_cols]

# Save
out_path = MERGED_DIR / "countries.csv"
countries_clean.to_csv(out_path, index=False)
print(f"✅ Saved: {out_path}")
print(f"   Shape : {countries_clean.shape[0]:,} rows × {countries_clean.shape[1]} cols")
print(f"   Columns: {list(countries_clean.columns)}")
countries_clean.head(5)

---
## Summary & Coverage Report

In [ ]:
print("=" * 60)
print("📊 MERGE SUMMARY")
print("=" * 60)

for name, df in [("cities.csv", cities_clean), ("countries.csv", countries_clean)]:
    total      = len(df)
    null_frac  = df.isnull().mean().mean()
    coverage   = df.notnull().mean().sort_values(ascending=False)
    print(f"\n📁 {name}")
    print(f"   Rows    : {total:,}")
    print(f"   Columns : {df.shape[1]}")
    print(f"   Overall NaN rate: {null_frac:.1%}")
    print(f"   Column coverage (top 10):")
    for col, cov in coverage.head(10).items():
        bar = "█" * int(cov * 20)
        print(f"     {col:<45} {cov:5.1%}  {bar}")